
# Notebook 7 — Human-in-the-Loop + Feedback & Memory

## Apple Plant Disease XAI Project

### Pipeline position

**Notebook 1:** Apple data preparation  
→ **Notebook 2:** Transfer learning + ResNet50  
→ **Notebook 3:** Grad-CAM  
→ **Notebook 4:** Trust + uncertainty estimation  
→ **Notebook 5:** RAG knowledge base  
→ **Notebook 6:** Grounded explanation with Qwen2.5-7B-Instruct  
→ **Notebook 7:** **Human-in-the-Loop + Feedback/Memory**  
→ **Notebook 8:** Final integration + dashboard  
→ Final evaluation

### Goal

This notebook converts the outputs of the previous notebooks into a persistent human-review and feedback layer.

For every test sample we keep:

- model prediction
- class probabilities
- confidence
- entropy and uncertainty
- trust score / trust level
- Grad-CAM activation statistic
- RAG-grounded LLM explanation
- human-review status
- optional human-confirmed/corrected label
- optional human comment
- timestamp
- feedback source

The notebook is designed to be **resume-safe** and saves all important artifacts to Google Drive.

> Important: the notebook can create **demo/simulated feedback records for pipeline testing**, but these are explicitly marked as `simulated`. They must not be presented as real expert annotations.



## 1. What this notebook produces

Main output directory:

`MyDrive/XAI_Project/results/human_feedback/`

Expected artifacts:

```text
human_feedback/
├── human_review_queue.csv
├── human_review_queue.jsonl
├── feedback_records.csv
├── feedback_records.jsonl
├── feedback_memory.csv
├── feedback_memory.jsonl
├── human_feedback_summary.json
├── human_feedback_config.json
├── notebook7_completion_manifest.json
└── errors.jsonl
```

### How Notebook 8 will use these outputs

Notebook 8 can use:

1. `human_review_queue.csv`
   - display low-trust/high-uncertainty cases
   - show prediction, probabilities, trust, Grad-CAM statistic and explanation

2. `feedback_records.csv`
   - store actual human decisions

3. `feedback_memory.csv`
   - retrieve previously reviewed examples
   - support future case-based explanations

4. `human_feedback_summary.json`
   - dashboard counters and project statistics


In [ ]:
# ============================================================
# CELL 1 — CHECK EXISTING COLAB ENVIRONMENT
# ============================================================

import sys

print("Python:", sys.version)

import numpy as np
import pandas as pd

print("NumPy :", np.__version__)
print("Pandas:", pd.__version__)

# Check SciPy only
try:
    import scipy
    print("SciPy :", scipy.__version__)
except Exception as e:
    print("SciPy import failed:", e)

print("\nEnvironment check completed.")

In [ ]:
# ============================================================
# CELL 2 — IMPORTS
# ============================================================

import os
import json
import math
import time
import traceback
import re
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from IPython.display import display, Markdown

print("All required imports completed.")
print("NumPy :", np.__version__)
print("Pandas:", pd.__version__)

try:
    import scipy
    print("SciPy :", scipy.__version__)
except Exception:
    pass

print("Notebook 7 environment is ready.")

In [ ]:

# ============================================================
# CELL 2 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/XAI_Project")
RESULTS_DIR = BASE_DIR / "results"

TRUST_DIR = RESULTS_DIR / "trust_uncertainty"
LLM_DIR = RESULTS_DIR / "llm_explanations"
HF_DIR = RESULTS_DIR / "human_feedback"

HF_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR :", BASE_DIR)
print("TRUST_DIR:", TRUST_DIR)
print("LLM_DIR  :", LLM_DIR)
print("HF_DIR   :", HF_DIR)


In [ ]:
# ============================================================
# CELL 3 — CONFIGURATION
# ============================================================

CLASS_NAMES = {
    0: "Apple Healthy",
    1: "Apple Scab",
    2: "Apple Black Rot",
    3: "Apple Cedar Apple Rust",
}
NUM_CLASSES = len(CLASS_NAMES)

PROB_COLS = {
    0: "prob_0_Apple_Healthy",
    1: "prob_1_Apple_Scab",
    2: "prob_2_Apple_Black_Rot",
    3: "prob_3_Apple_Cedar_Apple_Rust",
}

TRUST_REPORT_PATH = RESULTS_DIR / "trust_uncertainty" / "trust_uncertainty_test_report.csv"

FULL_LLM_DIR = RESULTS_DIR / "llm_explanations"
FULL_LLM_CSV = FULL_LLM_DIR / "llm_explanations_all_test_samples.csv"
FULL_LLM_JSONL = FULL_LLM_DIR / "llm_explanations_all_test_samples.jsonl"

PARTIAL_LLM_DIR = FULL_LLM_DIR / "llm_30_samples"
PARTIAL_LLM_CSV = PARTIAL_LLM_DIR / "llm_explanations_30_samples.csv"
PARTIAL_LLM_JSONL = PARTIAL_LLM_DIR / "llm_explanations_30_samples.jsonl"

if FULL_LLM_CSV.exists() or FULL_LLM_JSONL.exists():
    LLM_CSV_PATH, LLM_JSONL_PATH, LLM_OUTPUT_TYPE = FULL_LLM_CSV, FULL_LLM_JSONL, "full"
elif PARTIAL_LLM_CSV.exists() or PARTIAL_LLM_JSONL.exists():
    LLM_CSV_PATH, LLM_JSONL_PATH, LLM_OUTPUT_TYPE = PARTIAL_LLM_CSV, PARTIAL_LLM_JSONL, "partial_30_samples"
else:
    LLM_CSV_PATH, LLM_JSONL_PATH, LLM_OUTPUT_TYPE = FULL_LLM_CSV, FULL_LLM_JSONL, "missing"

TRUST_REVIEW_THRESHOLD = 0.70
CONFIDENCE_REVIEW_THRESHOLD = 0.80
ENTROPY_REVIEW_THRESHOLD = 0.50
MARGIN_REVIEW_THRESHOLD = 0.20
REVIEW_IF_ANY_CONDITION = True

CREATE_DEMO_FEEDBACK = True
DEMO_FEEDBACK_COUNT = 5
ALLOW_OVERWRITE_FEEDBACK = False

OUTPUT_FILES = {
    "review_queue_csv": HF_DIR / "human_review_queue.csv",
    "review_queue_jsonl": HF_DIR / "human_review_queue.jsonl",
    "feedback_csv": HF_DIR / "feedback_records.csv",
    "feedback_jsonl": HF_DIR / "feedback_records.jsonl",
    "memory_csv": HF_DIR / "feedback_memory.csv",
    "memory_jsonl": HF_DIR / "feedback_memory.jsonl",
    "summary_json": HF_DIR / "human_feedback_summary.json",
    "config_json": HF_DIR / "human_feedback_config.json",
    "manifest_json": HF_DIR / "notebook7_completion_manifest.json",
    "errors_jsonl": HF_DIR / "errors.jsonl",
    "evaluation_json": HF_DIR / "notebook7_evaluation_summary.json",
    "classification_metrics_csv": HF_DIR / "notebook7_classification_metrics.csv",
}

print("Configuration loaded.")
print("Notebook 6 output type:", LLM_OUTPUT_TYPE)
print("Notebook 6 CSV:", LLM_CSV_PATH)
print("Notebook 6 JSONL:", LLM_JSONL_PATH)



## 2. Helper functions

The following functions make the notebook robust to:

- JSONL files with nested dictionaries
- missing optional LLM fields
- empty/malformed lines
- duplicate sample IDs
- repeated notebook execution
- partially completed previous runs
- probability columns stored as strings


In [ ]:

# ============================================================
# CELL 4 — ROBUST FILE / JSON HELPERS
# ============================================================

def now_utc_iso():
    return datetime.now(timezone.utc).isoformat()

def json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if pd.isna(value) if not isinstance(value, (dict, list, tuple)) else False:
        return None
    return value

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=json_safe)

def append_jsonl(path, records):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False, default=json_safe) + "\n")

def read_jsonl(path):
    records = []
    path = Path(path)
    if not path.exists():
        return records

    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except Exception as e:
                print(f"Warning: invalid JSONL line {line_no}: {e}")
    return records

def flatten_llm_record(record):
    """Convert a Notebook 6 JSON record into stable scalar fields."""
    out = {}

    for key, value in record.items():
        if isinstance(value, (dict, list)):
            continue
        out[key] = value

    # Common nested structures used by Notebook 6.
    for nested_key in ["llm_output", "explanation", "generated_explanation"]:
        nested = record.get(nested_key)
        if isinstance(nested, dict):
            for key, value in nested.items():
                if isinstance(value, (dict, list)):
                    continue
                out[f"llm_{key}"] = value

    # Preserve complete structured explanation as JSON text.
    for candidate in ["llm_output", "explanation", "generated_explanation"]:
        if candidate in record:
            try:
                out["llm_explanation_json"] = json.dumps(
                    record[candidate], ensure_ascii=False, default=json_safe
                )
                break
            except Exception:
                pass

    return out

print("Helper functions ready.")


In [ ]:
# ============================================================
# CELL 5 — DISCOVER INPUT ARTIFACTS
# ============================================================

print("=" * 70)
print("INPUT ARTIFACT CHECK")
print("=" * 70)

print("Notebook 4:", TRUST_REPORT_PATH, "| exists:", TRUST_REPORT_PATH.exists())
print("Notebook 6:", LLM_CSV_PATH, "| exists:", LLM_CSV_PATH.exists())
print("Notebook 6:", LLM_JSONL_PATH, "| exists:", LLM_JSONL_PATH.exists())

if not TRUST_REPORT_PATH.exists():
    raise FileNotFoundError(
        f"Notebook 4 output not found:\n{TRUST_REPORT_PATH}\nRun Notebook 4 first."
    )

if not LLM_CSV_PATH.exists() and not LLM_JSONL_PATH.exists():
    raise FileNotFoundError(
        "Notebook 6 output not found. Expected either full output or current 30-sample output."
    )

print("\nAll required input artifacts are available.")

# Initialize the persistent error log if it does not exist.
OUTPUT_FILES["errors_jsonl"].parent.mkdir(parents=True, exist_ok=True)
if not OUTPUT_FILES["errors_jsonl"].exists():
    OUTPUT_FILES["errors_jsonl"].write_text("", encoding="utf-8")


In [ ]:

# ============================================================
# CELL 6 — LOAD NOTEBOOK 4 TRUST / UNCERTAINTY DATA
# ============================================================

trust_df = pd.read_csv(TRUST_REPORT_PATH)

print("Notebook 4 rows:", len(trust_df))
print("Notebook 4 columns:")
print(trust_df.columns.tolist())

required_trust_columns = [
    "image_index",
    "true_label",
    "true_class",
    "predicted_label",
    "predicted_class",
    "correct",
    "confidence",
    "entropy",
    "entropy_reliability",
    "margin",
    "margin_reliability",
    "calibrated_confidence",
    "trust_score",
    "trust_level",
    "gradcam_high_activation_area_percent",
]

missing = [c for c in required_trust_columns if c not in trust_df.columns]
if missing:
    raise ValueError(f"Notebook 4 report is missing required columns: {missing}")

# Numeric conversion.
numeric_cols = [
    "image_index",
    "true_label",
    "predicted_label",
    "confidence",
    "entropy",
    "entropy_reliability",
    "margin",
    "margin_reliability",
    "calibrated_confidence",
    "trust_score",
    "gradcam_high_activation_area_percent",
]

for col in numeric_cols:
    trust_df[col] = pd.to_numeric(trust_df[col], errors="coerce")

trust_df["image_index"] = trust_df["image_index"].astype("Int64")
trust_df["true_label"] = trust_df["true_label"].astype("Int64")
trust_df["predicted_label"] = trust_df["predicted_label"].astype("Int64")

# Validate probability columns.
for col in PROB_COLS.values():
    if col not in trust_df.columns:
        raise ValueError(f"Missing probability column: {col}")
    trust_df[col] = pd.to_numeric(trust_df[col], errors="coerce")

prob_matrix = trust_df[list(PROB_COLS.values())].to_numpy(dtype=float)
prob_sums = np.nansum(prob_matrix, axis=1)

if np.isnan(prob_matrix).any():
    raise ValueError("Probability table contains NaN values.")

if not np.allclose(prob_sums, 1.0, atol=1e-3):
    bad = np.where(np.abs(prob_sums - 1.0) > 1e-3)[0][:10]
    raise ValueError(
        f"Probability rows do not sum to 1 within tolerance. "
        f"Example bad row indices: {bad.tolist()}"
    )

if trust_df["image_index"].duplicated().any():
    duplicates = trust_df.loc[trust_df["image_index"].duplicated(), "image_index"].tolist()[:20]
    raise ValueError(f"Duplicate image_index values found: {duplicates}")

print("\nTrust data validation passed.")
display(trust_df.head(3))


In [ ]:
# ============================================================
# CELL 7 — LOAD NOTEBOOK 6 LLM OUTPUT
# ============================================================

print("=" * 70)
print("NOTEBOOK 6 LLM OUTPUT")
print("=" * 70)

if LLM_CSV_PATH.exists():
    llm_df = pd.read_csv(LLM_CSV_PATH)
    print("Loaded CSV:", LLM_CSV_PATH)
else:
    records = read_jsonl(LLM_JSONL_PATH)
    if not records:
        raise ValueError("Notebook 6 JSONL contains no valid records.")
    llm_df = pd.DataFrame(records)
    print("Loaded JSONL:", LLM_JSONL_PATH)

if llm_df.empty:
    raise ValueError("Notebook 6 output is empty.")

print("Rows:", len(llm_df))
print("Columns:", llm_df.columns.tolist())

if "image_index" in llm_df.columns:
    llm_df["image_index"] = pd.to_numeric(llm_df["image_index"], errors="coerce")
elif "sample_id" in llm_df.columns:
    llm_df["image_index"] = pd.to_numeric(
        llm_df["sample_id"].astype(str).str.extract(r"(\d+)", expand=False),
        errors="coerce"
    )
else:
    raise ValueError("Notebook 6 output contains neither image_index nor sample_id.")

invalid = int(llm_df["image_index"].isna().sum())
if invalid:
    print("Dropping invalid sample IDs:", invalid)
    llm_df = llm_df[llm_df["image_index"].notna()].copy()

llm_df["image_index"] = llm_df["image_index"].astype(int)
llm_df = llm_df.drop_duplicates("image_index", keep="last").reset_index(drop=True)

if "sample_id" not in llm_df.columns:
    llm_df["sample_id"] = llm_df["image_index"].apply(lambda x: f"test_{int(x):05d}")

LLM_FIELDS = [
    "summary", "prediction_explanation", "visual_evidence",
    "knowledge_evidence", "uncertainty_note",
    "human_review_recommendation", "source_references", "rag_sources_used"
]

def build_llm_explanation(row):
    payload = {}
    for field in LLM_FIELDS:
        if field in row.index:
            value = row[field]
            if not pd.isna(value):
                payload[field] = value
    return json.dumps(payload, ensure_ascii=False)

llm_df["llm_explanation_json"] = llm_df.apply(build_llm_explanation, axis=1)

print("\nSample ID mapping:")
display(llm_df[["sample_id", "image_index"]].head(10))
print("\nUnique LLM samples:", len(llm_df))


In [ ]:
# ============================================================
# CELL 8 — VERIFY ID ALIGNMENT
# ============================================================

trust_ids = set(trust_df["image_index"].astype(int))
llm_ids = set(llm_df["image_index"].astype(int))

overlap = trust_ids.intersection(llm_ids)
missing_llm = trust_ids - llm_ids
extra_llm = llm_ids - trust_ids

coverage = 100.0 * len(overlap) / max(len(trust_ids), 1)

print("=" * 70)
print("NOTEBOOK 4 ↔ NOTEBOOK 6 ALIGNMENT")
print("=" * 70)
print("Notebook 4 samples :", len(trust_ids))
print("Notebook 6 samples :", len(llm_ids))
print("Matching IDs       :", len(overlap))
print("Missing LLM IDs    :", len(missing_llm))
print("Extra LLM IDs      :", len(extra_llm))
print(f"LLM coverage       : {coverage:.2f}%")

if len(overlap) == 0:
    alignment_status = "no_overlap"
    print("\nWARNING: No matching IDs. LLM evidence will not be attached.")
elif len(overlap) < len(trust_ids):
    alignment_status = "partial_coverage"
    print("\nPartial LLM coverage detected.")
else:
    alignment_status = "full"
    print("\nFull LLM coverage detected.")

print("\nNotebook 4 first IDs:", sorted(trust_ids)[:10])
print("Notebook 6 first IDs:", sorted(llm_ids)[:10])

if missing_llm:
    print("First missing LLM IDs:", sorted(missing_llm)[:20])
if extra_llm:
    print("First extra LLM IDs:", sorted(extra_llm)[:20])


In [ ]:
# ============================================================
# CELL 9 — MERGE NOTEBOOK 4 + NOTEBOOK 6
# ============================================================

llm_merge_df = llm_df[
    llm_df["image_index"].isin(trust_ids)
].drop_duplicates("image_index", keep="last").copy()

work_df = trust_df.merge(
    llm_merge_df,
    on="image_index",
    how="left",
    suffixes=("", "_llm")
)

work_df["llm_available"] = work_df["image_index"].isin(
    set(llm_merge_df["image_index"].astype(int))
)

print("=" * 70)
print("MERGE COMPLETE")
print("=" * 70)
print("Notebook 4 samples :", len(trust_df))
print("Merged samples     :", len(work_df))
print("LLM available      :", int(work_df["llm_available"].sum()))
print("LLM missing        :", int((~work_df["llm_available"]).sum()))
print(f"LLM coverage       : {100.0 * work_df['llm_available'].sum() / max(len(work_df),1):.2f}%")

if len(work_df) != len(trust_df):
    raise RuntimeError("Merge changed the Notebook 4 row count.")

if not work_df["image_index"].is_unique:
    raise RuntimeError("Merged dataframe contains duplicate image_index values.")

display(work_df[
    ["image_index","true_class","predicted_class","confidence",
     "entropy","margin","trust_score","trust_level","llm_available"]
].head(10))



## 3. Human-review routing policy

A sample enters the review queue when one or more configurable conditions indicate that additional human inspection may be useful:

- low trust score
- low confidence
- high entropy
- low prediction margin

This is a **workflow threshold**, not a claim that a particular score means a disease is present or absent.

The thresholds can be changed in the configuration cell.


In [ ]:

# ============================================================
# CELL 9 — CREATE HUMAN REVIEW QUEUE
# ============================================================

def review_reasons(row):
    reasons = []

    trust = float(row["trust_score"])
    confidence = float(row["confidence"])
    entropy = float(row["entropy"])
    margin = float(row["margin"])

    if trust < TRUST_REVIEW_THRESHOLD:
        reasons.append("low_trust")

    if confidence < CONFIDENCE_REVIEW_THRESHOLD:
        reasons.append("low_confidence")

    if entropy > ENTROPY_REVIEW_THRESHOLD:
        reasons.append("high_entropy")

    if margin < MARGIN_REVIEW_THRESHOLD:
        reasons.append("low_margin")

    return reasons

work_df["review_reasons"] = work_df.apply(review_reasons, axis=1)
work_df["requires_human_review"] = work_df["review_reasons"].apply(
    lambda x: len(x) > 0
)

work_df["review_priority"] = (
    (1.0 - work_df["trust_score"].clip(0, 1))
    + (1.0 - work_df["confidence"].clip(0, 1))
    + work_df["entropy"].clip(lower=0)
    + (1.0 - work_df["margin"].clip(0, 1))
)

review_queue = work_df[work_df["requires_human_review"]].copy()

# Highest priority first.
review_queue = review_queue.sort_values(
    by=["review_priority", "trust_score"],
    ascending=[False, True],
).reset_index(drop=True)

print("Total samples:", len(work_df))
print("Review queue:", len(review_queue))
print("Review percentage:", round(100 * len(review_queue) / max(len(work_df), 1), 2), "%")

display(review_queue[[
    "image_index",
    "predicted_class",
    "confidence",
    "entropy",
    "margin",
    "trust_score",
    "trust_level",
    "gradcam_high_activation_area_percent",
    "review_reasons",
]].head(20))


In [ ]:

# ============================================================
# CELL 10 — CREATE HUMAN-REVIEW RECORDS
# ============================================================

def build_review_record(row):
    probabilities = {
        CLASS_NAMES[i]: float(row[PROB_COLS[i]])
        for i in range(NUM_CLASSES)
    }

    llm_explanation = row.get("llm_explanation_json", None)
    try:
        if pd.isna(llm_explanation):
            llm_explanation = None
    except Exception:
        pass

    return {
        "sample_id": f"test_{int(row['image_index']):05d}",
        "image_index": int(row["image_index"]),
        "true_label": int(row["true_label"]) if pd.notna(row["true_label"]) else None,
        "true_class": str(row["true_class"]),
        "model_prediction": int(row["predicted_label"]),
        "model_prediction_class": str(row["predicted_class"]),
        "model_confidence": float(row["confidence"]),
        "entropy": float(row["entropy"]),
        "margin": float(row["margin"]),
        "calibrated_confidence": float(row["calibrated_confidence"]),
        "trust_score": float(row["trust_score"]),
        "trust_level": str(row["trust_level"]),
        "gradcam_high_activation_area_percent": float(
            row["gradcam_high_activation_area_percent"]
        ),
        "probabilities": probabilities,
        "llm_explanation": llm_explanation,
        "llm_available": bool(row["llm_available"]),
        "requires_human_review": bool(row["requires_human_review"]),
        "review_reasons": list(row["review_reasons"]),
        "review_priority": float(row["review_priority"]),
        "human_decision": None,
        "corrected_class": None,
        "human_comment": None,
        "feedback_source": None,
        "review_timestamp": None,
        "feedback_version": 1,
    }

review_records = [build_review_record(row) for _, row in review_queue.iterrows()]

print("Review records created:", len(review_records))

if review_records:
    print(json.dumps(review_records[0], indent=2, ensure_ascii=False)[:5000])


In [ ]:

# ============================================================
# CELL 11 — SAVE HUMAN REVIEW QUEUE
# ============================================================

# JSONL
with open(OUTPUT_FILES["review_queue_jsonl"], "w", encoding="utf-8") as f:
    for record in review_records:
        f.write(json.dumps(record, ensure_ascii=False, default=json_safe) + "\n")

# CSV: flatten nested probability/explanation fields for easy dashboard loading.
review_csv_records = []

for record in review_records:
    flat = {k: v for k, v in record.items() if k != "probabilities"}
    flat["review_reasons"] = ";".join(record["review_reasons"])

    for class_id, class_name in CLASS_NAMES.items():
        flat[f"prob_{class_id}"] = record["probabilities"][class_name]

    review_csv_records.append(flat)

review_queue_df = pd.DataFrame(review_csv_records)

if not review_queue_df.empty:
    review_queue_df.to_csv(
        OUTPUT_FILES["review_queue_csv"],
        index=False,
        encoding="utf-8",
    )
else:
    # Keep an explicit empty schema.
    pd.DataFrame(columns=[
        "sample_id", "image_index", "true_label", "true_class",
        "model_prediction", "model_prediction_class", "model_confidence",
        "entropy", "margin", "calibrated_confidence", "trust_score",
        "trust_level", "gradcam_high_activation_area_percent",
        "llm_explanation", "llm_available", "requires_human_review",
        "review_reasons", "review_priority", "human_decision",
        "corrected_class", "human_comment", "feedback_source",
        "review_timestamp", "feedback_version",
    ]).to_csv(OUTPUT_FILES["review_queue_csv"], index=False)

print("Saved:")
print(OUTPUT_FILES["review_queue_csv"])
print(OUTPUT_FILES["review_queue_jsonl"])
